# Kubernetes API Documentation Agent

This notebook walks through the full pipeline for the Kubernetes API RAG system: it downloads and chunks the Kubernetes OpenAPI specification, ingests those chunks into a Vertex AI Search data store, and launches an interactive Gradio chat interface where you can ask natural-language questions about the Kubernetes API and receive answers grounded in the official documentation.

In [ ]:
!pip install -q uv
!uv pip install -q --system gradio google-cloud-discoveryengine vertexai requests pyyaml

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
GCP_PROJECT_ID = "your-project-id"               # @param {type:"string"}
GCP_LOCATION = "us-central1"                      # @param {type:"string"}
VERTEX_SEARCH_DATA_STORE_ID = "your-data-store-id"  # @param {type:"string"}
API_SPEC_URL = "https://raw.githubusercontent.com/kubernetes/kubernetes/master/api/openapi-spec/swagger.json"  # @param {type:"string"}
API_NAME = "Kubernetes"                           # @param {type:"string"}

import os
os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
os.environ["GCP_LOCATION"] = GCP_LOCATION
os.environ["VERTEX_SEARCH_DATA_STORE_ID"] = VERTEX_SEARCH_DATA_STORE_ID
os.environ["API_NAME"] = API_NAME

## Step 1: Ingest Kubernetes API Documentation

In [ ]:
!python -m src.ingest --spec {API_SPEC_URL} --name {API_NAME.lower()}

## Step 2: Upload to Vertex AI Search

After ingestion, a `data/kubernetes_chunks.jsonl` file will be available. To create a data store and upload this file:

1. Open the [Vertex AI Search console](https://console.cloud.google.com/gen-app-builder/engines).
2. Click **Create app**, choose **Search**, and select **Generic** content type.
3. Give your data store a name and note the generated **Data Store ID** — paste it into `VERTEX_SEARCH_DATA_STORE_ID` in Cell 4 above.
4. Under **Import data**, choose **Cloud Storage** or **Upload files**, then upload `data/kubernetes_chunks.jsonl`.
5. Wait for the import job to complete before proceeding to Step 3.

## Step 3: Launch the Documentation Agent

In [ ]:
import os
os.chdir("/content/api-rag")  # adjust if repo is cloned elsewhere

# Set env vars before importing app
os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
os.environ["GCP_LOCATION"] = GCP_LOCATION
os.environ["VERTEX_SEARCH_DATA_STORE_ID"] = VERTEX_SEARCH_DATA_STORE_ID

from src.app import demo
demo.launch(share=True)